# 03. Формирование аналитической таблицы и создание признаков

## 3.1. Цель этапа и основные задачи

## Цель

Сформировать единую аналитическую таблицу уровня заказа, в которой одна строка соответствует одному заказу, и создать производные признаки, необходимые для дальнейшего исследовательского анализа и построения моделей машинного обучения.

### Основные задачи

- определить уровень детализации исходных таблиц и подготовить необходимые агрегации;
- сформировать признаки на основе товарных, платёжных и географических данных;
- подготовить итоговую информацию об отзывах;
- объединить данные в единую таблицу уровня заказа;
- проконтролировать корректность объединений и сохранение принципа «одна строка — один заказ»;
- создать временные признаки, характеризующие момент покупки и этапы выполнения заказа;
- создать признаки соблюдения срока доставки;
- сформировать целевой признак `bad_review`;
- проверить корректность созданных признаков;
- сохранить итоговую аналитическую таблицу `orders_analytics` для дальнейшего EDA и моделирования.

## 3.2. Загрузка обработанных таблиц

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import load_processed_data

tables = load_processed_data(PROJECT_ROOT / "data" / "processed")
orders = tables["orders"]
customers = tables["customers"]
geolocation = tables["geolocation"]
order_items = tables["order_items"]
order_payments = tables["order_payments"]
order_reviews = tables["order_reviews"]
products = tables["products"]
sellers = tables["sellers"]
product_category_name_translation = tables["product_category_name_translation"]

In [3]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [4]:
order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  str           
 1   order_id                 99224 non-null  str           
 2   review_score             99224 non-null  int64         
 3   review_comment_title     11568 non-null  str           
 4   review_comment_message   40977 non-null  str           
 5   review_creation_date     99224 non-null  datetime64[us]
 6   review_answer_timestamp  99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(4)
memory usage: 5.3 MB


In [5]:
geolocation.duplicated().sum()

np.int64(0)

## 3.3. Уровень детализации каждой таблицы

| Таблица | Одна строка соответствует | Ключ | Нужна агрегация |
|---|---|---|---|
| `orders` | одному заказу | `order_id` | нет |
| `customers` | одному идентификатору покупателя для заказа | `customer_id` | нет |
| `order_items` | одной позиции товара в заказе | `order_id + order_item_id` | да |
| `order_payments` | одной записи об оплате | `order_id + payment_sequential` | да |
| `order_reviews` | одной записи об отзыве | `review_id` | нужно проверить |
| `products` | одному товару | `product_id` | нет |
| `sellers` | одному продавцу | `seller_id` | нет |
| `geolocation` | одной географической записи | уникального ключа нет | да |
| `product_category_name_translation` | одному переводу категории | `product_category_name` | нет |

### Вывод

Итоговая аналитическая таблица должна иметь уровень детализации «одна строка — один заказ». Таблица `orders` уже соответствует этому уровню и будет использоваться как основа.

Таблицы `order_items` и `order_payments` содержат несколько строк для одного заказа, поэтому перед объединением их необходимо агрегировать по `order_id`.

Для таблицы `order_reviews` необходимо проверить наличие нескольких записей на один заказ. Таблицу `geolocation` нужно предварительно привести к одной строке на почтовый индекс.

## 3.4. Подготовка информации о товарных позициях

### 3.4.1 Проверка ключей справочных таблиц

In [6]:
products["product_id"].is_unique

True

In [7]:
sellers["seller_id"].is_unique

True

In [8]:
product_category_name_translation["product_category_name"].is_unique

True

### 3.4.2. Добавление характеристик товаров

In [9]:
rows_before = len(order_items)
price_before = order_items["price"].sum()
freight_before = order_items["freight_value"].sum()

In [10]:
item_details = order_items.merge(
    products,
    on="product_id",
    how="left",
    validate="m:1",
    indicator=True
)

In [11]:
item_details.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_merge
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,both
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,both
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,both
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,both
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,both


In [12]:
(item_details["_merge"] == "left_only").sum()

np.int64(0)

In [13]:
item_details = item_details.drop(columns="_merge")

In [14]:
rows_after = len(item_details)
print("Строк до объединения: ", rows_before)
print("Строк после объединения: ", rows_after)
assert rows_before == rows_after

Строк до объединения:  112650
Строк после объединения:  112650


In [15]:
item_details[["order_id", "order_item_id"]].duplicated().sum()

assert not item_details.duplicated(
    subset=["order_id", "order_item_id"]
).any()

In [16]:
price_after = order_items["price"].sum()
freight_after = order_items["freight_value"].sum()

print("Сумма цен до:", price_before)
print("Сумма цен после:", price_after)

print("Сумма доставки до:", freight_before)
print("Сумма доставки после:", freight_after)

assert np.isclose(price_before, price_after)
assert np.isclose(freight_before, freight_after)

Сумма цен до: 13591643.7
Сумма цен после: 13591643.7
Сумма доставки до: 2251909.54
Сумма доставки после: 2251909.54


In [17]:
print("Объединение выполнено корректно.")

Объединение выполнено корректно.


Таблица `products` была присоединена к `order_items` по ключу `product_id` с использованием левого объединения.
Это позволило сохранить все товарные позиции, включая возможные позиции без найденных характеристик товара.
После объединения количество строк и финансовые суммы не изменились, а составной ключ `order_id + order_item_id` остался уникальным.
Следовательно, размножения или потери товарных позиций не произошло.

### 3.4.3. Добавление переводов категорий

In [18]:
item_details = item_details.merge(
    product_category_name_translation,
    on="product_category_name",
    how="left",
    validate="m:1",
    indicator=True
)

In [19]:
item_details["_merge"].value_counts()

_merge
both          111023
left_only       1627
right_only         0
Name: count, dtype: int64

In [20]:
item_details.loc[
    item_details["_merge"] == "left_only",
    ["product_id", "product_category_name"]
].drop_duplicates()

,product_id,product_category_name
123,ff6caf9340512b8bf6d2a2a6df032cfa,NaN
125,a9c404971d1a5b1cbc2e4070e02731fd,NaN
132,5a848e4ab52fd5445cdc07aab1c40e48,NaN
142,41eee23c25f7a574dfaf8d5c151dbb12,NaN
171,e10758160da97891c2fdcbc35f0f031d,NaN
...,...,...
111914,28209b7aed95f7c0c9b9806f974a104e,NaN
112010,41bf6a8c9424d4ec060e39e21883db3f,NaN
112062,40a1e3c65a0bcf6f4ebba840a8156ba2,NaN
112091,34a4076cd057a823247a5a44e84872d8,NaN


In [21]:
item_details.loc[
    (item_details["_merge"] == "left_only") & (item_details["product_category_name"].notna()),
    ["product_id", "product_category_name"]
].drop_duplicates()

,product_id,product_category_name
3228,a4756663d007b0cd1af865754d08d968,portateis_cozinha_e_preparadores_de_alimentos
12976,6727051471a0fc4a0e7737b57bff2549,pc_gamer
13025,cb9d764f38ee4d0c00af64d5c388f837,portateis_cozinha_e_preparadores_de_alimentos
18629,dbe520fb381ad695a7e1f2807d20c765,pc_gamer
19702,c7a3f1a7f9eef146cc499368b578b884,portateis_cozinha_e_preparadores_de_alimentos
31806,0105b5323d24fc655f73052694dbbb3a,pc_gamer
36976,7afdd65f79f63819ff5bee328843fa37,portateis_cozinha_e_preparadores_de_alimentos
37083,bed164d9d628cf0593003389c535c6e0,portateis_cozinha_e_preparadores_de_alimentos
62090,1954739d84629e7323a4295812a3e0ec,portateis_cozinha_e_preparadores_de_alimentos
73369,ae62bb0f95af63d64eae5f93dddea8d3,portateis_cozinha_e_preparadores_de_alimentos


In [22]:
item_details.loc[
        (item_details["_merge"] == "left_only")
        & item_details["product_category_name"].notna(),
        "product_category_name"
].value_counts()

product_category_name
portateis_cozinha_e_preparadores_de_alimentos    15
pc_gamer                                          9
Name: count, dtype: int64

In [23]:
manual_translations = {
    "pc_gamer": "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_and_food_preparation_appliances"
}

In [24]:
item_details["product_category_name_english"] = item_details["product_category_name_english"].fillna(item_details["product_category_name"].map(manual_translations))

In [25]:
item_details.loc[
    item_details["product_category_name"].notna()
    & item_details["product_category_name_english"].isna(),
    ["product_category_name", "product_category_name_english"]
].drop_duplicates()

,product_category_name,product_category_name_english


In [26]:
item_details["category_name"] = (
    item_details["product_category_name_english"]
    .fillna("unknown")
)

In [27]:
item_details["category_name"].isna().sum()

np.int64(0)

In [28]:
assert len(item_details) == len(order_items)

assert not item_details.duplicated(
    subset=["order_id", "order_item_id"]
).any()

assert np.isclose(
    item_details["price"].sum(),
    order_items["price"].sum()
)

assert np.isclose(
    item_details["freight_value"].sum(),
    order_items["freight_value"].sum()
)

In [29]:
item_details = item_details.drop(columns="_merge")

In [30]:
item_details.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,category_name
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,garden_tools


После присоединения таблицы переводов для части товарных позиций перевод категории не был найден. У 1603 позиций отсутствовало исходное название категории. Кроме того, в таблице переводов отсутствовали две существующие категории: pc_gamer и portateis_cozinha_e_preparadores_de_alimentos. Для них переводы были добавлены вручную. Для позиций без исходной категории создано обозначение unknown. После обработки все позиции получили итоговое значение category_name, при этом количество строк и финансовые показатели не изменились.

### 3.4.4. Добавление информации о продавцах

In [31]:
rows_before = len(item_details)

In [32]:
item_details = item_details.merge(
    sellers,
    on="seller_id",
    how="left",
    validate="m:1",
    indicator=True
)

In [33]:
assert (item_details["_merge"] == "left_only").sum() == 0

In [34]:
assert len(item_details) == rows_before

In [35]:
assert item_details[
    [
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
].isna().sum().sum() == 0

In [36]:
assert not item_details.duplicated(
    subset=["order_id", "order_item_id"]
).any()

In [37]:
item_details = item_details.drop(columns="_merge")

In [38]:
item_details.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,category_name,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,cool_stuff,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,pet_shop,3471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,furniture_decor,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,perfumery,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,garden_tools,87900,loanda,PR


Таблица `sellers` была присоединена по ключу `seller_id` к таблице `item_details` с использованием левого объединения. После объединения количество строк в таблице `item_details` не изменилось, для всех `seller_id` была найдена информация из таблицы `sellers`, новые пропуски не появились, составной ключ `order_id + order_item_id` остался уникальным.

### 3.4.5. Проверка результата

In [39]:
assert price_before == item_details["price"].sum()
assert freight_before == item_details["freight_value"].sum()

На основе таблицы order_items сформирована промежуточная таблица товарных позиций. К каждой позиции добавлены характеристики товара, перевод категории и информация о продавце. Уровень детализации остался неизменным: одна строка соответствует одной позиции товара в заказе. Количество строк и финансовые суммы после объединений не изменились. На следующем шаге полученная таблица будет агрегирована до уровня одного заказа.

## 3.5. Агрегация товарных позиций до уровня заказа

### 3.5.1. Выбор агрегируемых признаков

| Новый столбец | Что означает |
|---|---|
| `items_count` | количество товарных позиций |
| `products_count` | количество уникальных товаров |
| `sellers_count` | количество уникальных продавцов |
| `categories_count` | количество известных товарных категорий |
| `total_price` | общая стоимость товаров |
| `mean_item_price` | средняя цена позиции |
| `max_item_price` | максимальная цена позиции |
| `total_freight` | общая стоимость доставки |
| `mean_freight` | средняя стоимость доставки одной позиции |
| `total_weight_g` | суммарный известный вес товарных позиций, г |
| `total_volume_cm3` | суммарный расчётный объём товарных позиций, см³ |
| `has_unknown_category` | наличие неизвестной категории в заказе |
| `has_multiple_sellers` | наличие нескольких продавцов в заказе |
| `has_multiple_products` | наличие нескольких уникальных товаров в заказе |
| `order_total` | общая стоимость заказа с доставкой |
| `freight_ratio` | доля доставки в общей стоимости |

### 3.5.2. Сохранение контрольных показателей

In [40]:
rows_before_aggregation = len(item_details)
orders_before_aggregation = item_details["order_id"].nunique()
price_before_aggregation = item_details["price"].sum()
freight_before_aggregation = item_details["freight_value"].sum()

### 3.5.3. Основная агрегация по order_id

In [41]:
item_details["product_volume_cm3"] = (item_details["product_height_cm"] * item_details["product_width_cm"] * item_details["product_length_cm"])

order_items_agg = (
    item_details.groupby("order_id", as_index=False)
    .agg(
        items_count=("order_item_id", "size"),
        products_count=("product_id", "nunique"),
        sellers_count=("seller_id", "nunique"),
        categories_count=("product_category_name_english", "nunique"),
        unknown_category_items=("product_category_name_english", lambda column: column.isna().sum()),
        total_price=("price", "sum"),
        mean_item_price=("price", "mean"),
        max_item_price=("price", "max"),
        total_freight=("freight_value", "sum"),
        mean_freight=("freight_value", "mean"),
        total_weight_g=("product_weight_g", lambda column: column.sum(min_count=1)),
        total_volume_cm3=("product_volume_cm3", lambda column: column.sum(min_count=1))
    )
)

### 3.5.4. Создание дополнительных признаков заказа

In [42]:
order_items_agg["has_unknown_category"] = (order_items_agg["unknown_category_items"] > 0).astype(int)

order_items_agg["has_multiple_sellers"] = (
    order_items_agg["sellers_count"] > 1
).astype(int)

order_items_agg["has_multiple_products"] = (
    order_items_agg["products_count"] > 1
).astype(int)

order_items_agg["order_total"] = (
    order_items_agg["total_price"]
    + order_items_agg["total_freight"]
)

assert not (order_items_agg["order_total"] == 0).sum()

order_items_agg["freight_ratio"] = (order_items_agg["total_freight"] / order_items_agg["order_total"])

### 3.5.5. Проверка результата агрегации

In [43]:
assert len(order_items_agg) == orders_before_aggregation

assert order_items_agg["order_id"].is_unique

assert (
    order_items_agg["items_count"].sum()
    == rows_before_aggregation
)

In [44]:
order_items_agg["items_count"].describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: items_count, dtype: float64

In [45]:
assert order_items_agg["items_count"].min() >= 1

In [46]:
assert np.isclose(
    order_items_agg["total_price"].sum(),
    price_before_aggregation
)

assert np.isclose(
    order_items_agg["total_freight"].sum(),
    freight_before_aggregation
)

assert order_items_agg["freight_ratio"].between(0, 1).all()

In [47]:
order_items_agg.head()

,order_id,items_count,products_count,sellers_count,categories_count,unknown_category_items,total_price,mean_item_price,max_item_price,total_freight,mean_freight,total_weight_g,total_volume_cm3,has_unknown_category,has_multiple_sellers,has_multiple_products,order_total,freight_ratio
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,1,0,58.90,58.90,58.90,13.29,13.29,650.0,3528.0,0,0,0,72.19,0.184098
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,1,0,239.90,239.90,239.90,19.93,19.93,30000.0,60000.0,0,0,0,259.83,0.076704
2,000229ec398224ef6ca0657da4fc703e,1,1,1,1,0,199.00,199.00,199.00,17.87,17.87,3050.0,14157.0,0,0,0,216.87,0.082400
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,1,0,12.99,12.99,12.99,12.79,12.79,200.0,2400.0,0,0,0,25.78,0.496121
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,1,0,199.90,199.90,199.90,18.14,18.14,3750.0,42000.0,0,0,0,218.04,0.083196


In [48]:
order_items_agg.shape

(98666, 18)

In [49]:
order_items_agg.isna().sum()

order_id                   0
items_count                0
products_count             0
sellers_count              0
categories_count           0
unknown_category_items     0
total_price                0
mean_item_price            0
max_item_price             0
total_freight              0
mean_freight               0
total_weight_g            16
total_volume_cm3          16
has_unknown_category       0
has_multiple_sellers       0
has_multiple_products      0
order_total                0
freight_ratio              0
dtype: int64

In [50]:
order_items_agg[
    [
        "total_price",
        "mean_item_price",
        "max_item_price",
        "total_freight",
        "mean_freight",
        "order_total"
    ]
].lt(0).sum()

total_price        0
mean_item_price    0
max_item_price     0
total_freight      0
mean_freight       0
order_total        0
dtype: int64

In [51]:
assert (
    order_items_agg["products_count"]
    <= order_items_agg["items_count"]
).all()

assert (
    order_items_agg["sellers_count"]
    <= order_items_agg["items_count"]
).all()

Подготовленные товарные позиции были сгруппированы по `order_id`. Для каждого заказа рассчитаны количество позиций, уникальных товаров, продавцов и категорий, а также показатели стоимости товаров и доставки. После агрегации `order_id` стал уникальным, количество итоговых строк совпало с количеством уникальных заказов, а общие суммы `price` и `freight_value` сохранились. Полученная таблица `order_items_agg` соответствует уровню детализации «одна строка - один заказ» и готова к дальнейшему объединению с таблицей `orders`.

## 3.6. Агрегация платежей до уровня заказа

### 3.6.1. Исследование структуры платежей

In [52]:
order_payments.groupby("order_id").size().describe()

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64

In [53]:
(
    order_payments.groupby("order_id").size() > 1
).sum()

np.int64(2961)

In [54]:
order_payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Для большинства заказов имеется одна платёжная запись: медиана и первые три квартиля равны единице.

### 3.6.2. Сохранение контрольных показателей

In [55]:
payment_rows_before = len(order_payments)
payment_orders_before = order_payments["order_id"].nunique()
payment_value_before = order_payments["payment_value"].sum()

### 3.6.3. Основная агрегация платежей

In [56]:
order_payments_agg = (
    order_payments.groupby("order_id", as_index=False)
    .agg(
        payment_total=("payment_value", "sum"),
        payments_count=("payment_sequential", "size"),
        payment_types_count=("payment_type", "nunique"),
        max_installments=("payment_installments", "max")
    )
)

### 3.6.4. Определение основного способа оплаты

In [57]:
#суммируем платехи по заказу и способу оплаты
payment_type_totals = (
    order_payments.groupby(["order_id", "payment_type"], as_index=False)
    .agg(payment_type_value=("payment_value", "sum"))
)

#для каждого заказа выбираем строку с максимальной суммой
main_payment_type = (
    payment_type_totals.sort_values(
        ["order_id", "payment_type_value"],
        ascending=[True, False]
        )
        .drop_duplicates(
            subset="order_id",
            keep="first"
        )
        .rename(
            columns={"payment_type": "main_payment_type"}
        )[["order_id", "main_payment_type"]]
)

#присоединить результат к основной агрегированной таблице
order_payments_agg = order_payments_agg.merge(
    main_payment_type,
    on="order_id",
    how="left",
    validate="1:1"
)

Основным способом оплаты выбран способ, на который приходится наибольшая суммарная часть оплаты заказа. Если заказ оплачивался несколькими записями одного типа, их суммы предварительно складывались.

### 3.6.5. Создание дополнительных признаков

In [58]:
#индикатор использования нескольких пособов оплаты
order_payments_agg["has_multiple_payment_types"] = (
    order_payments_agg["payment_types_count"] > 1
).astype(int)

#индикатор рассрочки
order_payments_agg["used_installments"] = (
    order_payments_agg["max_installments"] > 1
).astype(int)

### 3.6.6. Проверка результата агрегации

In [59]:
#количество строк
assert len(order_payments_agg) == payment_orders_before

#уникальность заказа
assert order_payments_agg["order_id"].is_unique

#сохранение общей суммы
assert np.isclose(
    order_payments_agg["payment_total"].sum(),
    payment_value_before
)

#все платёжные записи учтены
assert (
    order_payments_agg["payments_count"].sum()
    == payment_rows_before
)

#различных способов оплаты не может быть больше, чем платёжных записей
assert (
    order_payments_agg["payment_types_count"]
    <= order_payments_agg["payments_count"]
).all()

In [60]:
order_payments_agg.isna().sum()

order_id                      0
payment_total                 0
payments_count                0
payment_types_count           0
max_installments              0
main_payment_type             0
has_multiple_payment_types    0
used_installments             0
dtype: int64

In [61]:
(order_payments_agg["payment_total"] < 0).sum()

np.int64(0)

In [62]:
order_payments_agg.head()

,order_id,payment_total,payments_count,payment_types_count,max_installments,main_payment_type,has_multiple_payment_types,used_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1,2,credit_card,0,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1,3,credit_card,0,1
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1,5,credit_card,0,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1,2,credit_card,0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1,3,credit_card,0,1


### 3.6.7. Поиск заказов без платежей

In [63]:
orders_without_payments = orders.loc[
    ~orders["order_id"].isin(order_payments_agg["order_id"]),
    ["order_id", "order_status"]
]

In [64]:
len(orders_without_payments)

1

In [65]:
orders_without_payments["order_status"].value_counts()

order_status
delivered    1
Name: count, dtype: int64

In [66]:
orders.loc[
    orders["order_id"].isin(orders_without_payments["order_id"])
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04


In [67]:
order_items.loc[
    order_items["order_id"].isin(
        orders_without_payments["order_id"]
    )
]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
84389,bfbd0f9bdef84302105ad712db648a6c,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83
84390,bfbd0f9bdef84302105ad712db648a6c,2,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83
84391,bfbd0f9bdef84302105ad712db648a6c,3,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83


При сравнении таблиц `orders` и `order_payments_agg` был обнаружен один заказ без платёжных записей. При этом заказ имеет статус `delivered`, поэтому отсутствие платежа, вероятнее всего, связано с неполнотой или несогласованностью данных, а не с фактической бесплатной доставкой. Заказ сохранён для дальнейшего анализа; после объединения отсутствие платёжной информации будет отмечено пропусками и отдельным индикатором.

Платёжные записи были сгруппированы по `order_id`, в результате чего сформирована таблица ` order_payments_agg` с уровнем детализации «одна строка - один заказ». Для каждого заказа рассчитаны общая сумма платежей, количество платёжных записей, количество использованных способов оплаты, максимальное число платежей в рассрочку и основной способ оплаты.

После агрегации `order_id` стал уникальным, количество строк совпало с количеством уникальных заказов в исходной таблице `order_payments`, а общая сумма `payment_value` сохранилась. Это подтверждает, что при группировке платёжные записи не были потеряны или продублированы.

## 3.7. Агрегация отзывов до уровня заказа

### 3.7.1. Исследование количества отзывов на заказ

In [68]:
#уникален ли order_id в order_reviews
order_reviews["order_id"].is_unique

False

In [69]:
#количество отзывов на каждый заказ
reviews_per_order = order_reviews.groupby("order_id").size()

In [70]:
reviews_per_order.describe()

count    98673.000000
mean         1.005584
std          0.075060
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
dtype: float64

In [71]:
reviews_per_order.value_counts().sort_index()

1    98126
2      543
3        4
Name: count, dtype: int64

In [72]:
#сколько заказов имеют больше одного отзыва
(reviews_per_order > 1).sum()

np.int64(547)

### 3.7.2. Выделение заказов с повторными отзывами

In [73]:
#order_id заказов с несколькими отзывами
repeated_order_ids = reviews_per_order[reviews_per_order > 1].index

In [74]:
repeated_reviews = order_reviews.loc[
    order_reviews["order_id"].isin(repeated_order_ids)
].copy()

### 3.7.3. Сравнение оценок в повторных отзывах

In [75]:
reviews_check = (
    order_reviews.groupby("order_id")
    .agg(
        reviews_count=("review_id", "size"),
        unique_scores_count=("review_score", "nunique")
    )
)

repeated_reviews_check = reviews_check.loc[
    reviews_check["reviews_count"] > 1
]

different_scores_count = (
    repeated_reviews_check["unique_scores_count"] > 1
).sum()

print(
    "Заказов с повторными отзывами:",
    len(repeated_reviews_check)
)

print(
    "Из них с разными оценками:",
    different_scores_count
)

Заказов с повторными отзывами: 547
Из них с разными оценками: 202


#### 3.7.4. Примеры заказов с различающимися оценками

In [76]:
orders_with_different_scores = repeated_reviews_check.loc[
    repeated_reviews_check["unique_scores_count"] > 1
].index

order_reviews_with_diff_scores = order_reviews.loc[
    order_reviews["order_id"].isin(orders_with_different_scores),
    [
        "order_id",
        "review_id",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp"
    ]
].sort_values(
    ["order_id", "review_creation_date", "review_answer_timestamp"]
)

In [77]:
order_reviews_with_diff_scores

,order_id,review_id,review_score,review_creation_date,review_answer_timestamp
22779,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,2018-02-22,2018-02-23 12:12:30
68633,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,2018-03-04,2018-03-05 17:02:00
89888,02355020fd0a40a0d56df9f6ff060413,0c8e7347f1cdd2aede37371543e3d163,3,2018-03-21,2018-03-22 01:32:08
17582,02355020fd0a40a0d56df9f6ff060413,017f0e1ea6386de662cbeba299c59ad1,1,2018-03-29,2018-03-30 03:16:19
37911,029863af4b968de1e5d6a82782e662f5,04d945e95c788a3aa1ffbee42105637b,5,2017-07-14,2017-07-17 13:58:06
...,...,...,...,...,...
70962,fcde2f7449493b4f207a23fae117b0fb,ca2fb34d8c9fc880f0f94dde4b17e8e3,4,2017-06-14,2017-06-17 15:18:36
1157,fd61441ba2a7b57e6342862e779b10b0,3c625d52413314975e47e144fbc3cb8e,5,2017-10-04,2017-10-05 18:02:50
86641,fd61441ba2a7b57e6342862e779b10b0,24ad2fc85ec000ba4fbfd9841d9a1972,4,2017-10-14,2017-10-16 21:23:28
94504,ff763b73e473d03c321bcd5a053316e8,c56a88a404315a0d9e412c1472dda2c4,5,2017-11-01,2017-11-01 19:55:52


### 3.7.5. Анализ временных меток повторных отзывов

In [78]:
repeated_reviews_summary = (
    repeated_reviews
    .groupby("order_id")
    .agg(
        reviews_count=("review_id", "size"),
        unique_review_ids=("review_id", "nunique"),
        unique_scores_count=("review_score", "nunique"),
        unique_creation_dates=("review_creation_date", "nunique"),
        unique_answer_dates=("review_answer_timestamp", "nunique"),
        missing_creation_dates=(
            "review_creation_date",
            lambda column: column.isna().sum()
        ),
        missing_answer_dates=(
            "review_answer_timestamp",
            lambda column: column.isna().sum()
        )
    )
)

repeated_reviews_summary["different_scores"] = (
    repeated_reviews_summary["unique_scores_count"] > 1
)

repeated_reviews_summary["different_creation_dates"] = (
    repeated_reviews_summary["unique_creation_dates"] > 1
)

repeated_reviews_summary["different_answer_dates"] = (
    repeated_reviews_summary["unique_answer_dates"] > 1
)

repeated_reviews_summary[
    [
        "different_scores",
        "different_creation_dates",
        "different_answer_dates"
    ]
].value_counts()

different_scores  different_creation_dates  different_answer_dates
False             True                      True                      220
True              True                      True                      172
False             False                     True                      125
True              False                     True                       30
Name: count, dtype: int64

In [79]:
repeated_reviews_summary[
    ["missing_creation_dates", "missing_answer_dates"]
].sum()

missing_creation_dates    0
missing_answer_dates      0
dtype: int64

### 3.7.6. Проверка корректности временных меток

In [80]:
#проверка логичности временных меток
invalid_review_dates = order_reviews.loc[
    order_reviews["review_answer_timestamp"]
    < order_reviews["review_creation_date"]
]

assert invalid_review_dates.empty

### 3.7.7. Правило выбора итогового отзыва

Итоговым отзывом считается запись с наиболее поздним `review_answer_timestamp`, поскольку она отражает последнее зафиксированное мнение покупателя. При совпадении времени ответа выбирается запись с наиболее поздней `review_creation_date`. Если обе даты совпадают, сохраняется последняя запись в исходной таблице.

### 3.7.8. Создание дополнительных признаков отзывов

In [81]:
reviews_prepared = order_reviews.copy()

#наличие текстового комментария
reviews_prepared["has_review_comment"] = (
    reviews_prepared["review_comment_message"]
    .fillna("")
    .str.strip()
    .ne("")
    .astype(int)
)

#наличие заголовка
reviews_prepared["has_review_title"] = (
    reviews_prepared["review_comment_title"]
    .fillna("")
    .str.strip()
    .ne("")
    .astype(int)
)

#количество записей на заказ
reviews_count = (
    reviews_prepared
    .groupby("order_id")
    .size()
    .rename("reviews_count")
)

reviews_prepared = reviews_prepared.merge(
    reviews_count,
    on="order_id",
    how="left",
    validate="m:1"
)

#индикатор повторных записей
reviews_prepared["had_multiple_reviews"] = (
    reviews_prepared["reviews_count"] > 1
).astype(int)


### 3.7.9. Формирование итоговой таблицы отзывов

In [82]:
#выбор последнего отзыва
reviews_prepared["_row_order"] = np.arange(
    len(reviews_prepared)
)

reviews_prepared = reviews_prepared.sort_values(
    [
        "order_id",
        "review_answer_timestamp",
        "review_creation_date",
        "_row_order"
    ],
    ascending=True,
    na_position="first",
    kind="stable"
)

order_reviews_final = (
    reviews_prepared
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
    [
        [
            "order_id",
            "review_id",
            "review_score",
            "review_creation_date",
            "review_answer_timestamp",
            "has_review_comment",
            "has_review_title",
            "reviews_count",
            "had_multiple_reviews"
        ]
    ]
    .reset_index(drop=True)
)

### 3.7.10. Проверка итоговой таблицы

In [83]:
#уникальность заказов
assert order_reviews_final["order_id"].is_unique

#количество строк
assert len(order_reviews_final) == (
    order_reviews["order_id"].nunique()
)

#допустимый диапазон оценок
assert order_reviews_final[
    "review_score"
].between(1, 5).all()

#минимальное количество отзывов
assert (
    order_reviews_final["reviews_count"] >= 1
).all()

#согласованность индикатора
assert (
    order_reviews_final["had_multiple_reviews"]
    == (
        order_reviews_final["reviews_count"] > 1
    ).astype(int)
).all()

#количество заказов с повторными отзывами
assert (
    order_reviews_final["had_multiple_reviews"].sum()
    == 547
)

#полные дубликаты
assert not order_reviews_final.duplicated().any()

### 3.7.11. Проверка заказов без отзывов

In [84]:
orders_without_reviews = orders.loc[
    ~orders["order_id"].isin(
        order_reviews_final["order_id"]
    ),
    ["order_id", "order_status"]
]
len(orders_without_reviews)

768

In [85]:
orders_without_reviews[
    "order_status"
].value_counts()

order_status
delivered      646
shipped         75
canceled        20
unavailable     14
processing       6
invoiced         5
created          2
Name: count, dtype: int64

### 3.7.12. Вывод

В таблице `order_reviews` содержатся отзывы для 98 673 уникальных заказов. Для 98 126 заказов имеется одна запись об отзыве, для 543 заказов — две записи, а для 4 заказов — три записи. Таким образом, 547 заказов имеют повторные отзывы, поэтому `order_id` в исходной таблице не является уникальным.

У 202 заказов с повторными отзывами оценки различаются, тогда как у остальных 345 заказов повторные записи содержат одинаковые оценки. Это показывает, что произвольное удаление повторных строк могло бы привести к потере информации об изменении мнения покупателя.

У всех 547 заказов с повторными отзывами различается `review_answer_timestamp`. При этом у 392 заказов различаются также даты создания опроса, а у 155 заказов дата создания совпадает, но время фактического ответа отличается. Полученные результаты подтверждают, что повторные строки во многих случаях отражают отдельные ответы покупателя, оставленные в разные моменты времени, а не являются простыми полными дубликатами.

Записей, в которых `review_answer_timestamp` предшествует `review_creation_date`, не обнаружено. В качестве итогового отзыва для каждого заказа была выбрана запись с наиболее поздним `review_answer_timestamp`, поскольку она отражает последнее зафиксированное мнение покупателя. При совпадении времени ответа использовалась более поздняя `review_creation_date`, а при полном совпадении временных меток сохранялась последняя строка исходной таблицы.

Дополнительно были созданы признаки наличия текстового комментария, наличия заголовка, количества записей на заказ и наличия повторных отзывов. После обработки сформирована таблица `order_reviews_final`, содержащая одну строку для каждого из 98 673 заказов с отзывом. В итоговой таблице `order_id` уникален, полные дубликаты отсутствуют, а оценки находятся в допустимом диапазоне от 1 до 5.

При сравнении с таблицей `orders` обнаружено 768 заказов без отзывов. Среди них 646 заказов имеют статус `delivered`, 75 — `shipped`, 20 — `canceled`, 14 — `unavailable`, остальные находятся в других статусах. Отсутствие отзыва не считается положительной оценкой: при последующем левом объединении такие заказы будут сохранены с пропущенным значением `review_score`.

Полученная таблица `order_reviews_final` соответствует уровню детализации «одна строка — один заказ с отзывом» и готова к объединению с основной таблицей заказов.

## 3.8. Подготовка географических данных

### 3.8.1. Проверка структуры таблицы geolocation


In [86]:
geolocation.info()

<class 'pandas.DataFrame'>
RangeIndex: 738332 entries, 0 to 738331
Data columns (total 5 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   geolocation_zip_code_prefix  738332 non-null  int64  
 1   geolocation_lat              738332 non-null  float64
 2   geolocation_lng              738332 non-null  float64
 3   geolocation_city             738332 non-null  str    
 4   geolocation_state            738332 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 28.2 MB


In [87]:
#уникален ли geolocation_zip_code_prefix
geolocation["geolocation_zip_code_prefix"].is_unique

False

In [88]:
#количество уникальных почтовых индексов
geolocation["geolocation_zip_code_prefix"].nunique()

19015

In [89]:
#сколько строк в среднем приходится на один почтовый индекс?
#какое максимальное количество географических записей приходится на один индекс?
geolocation.groupby("geolocation_zip_code_prefix").size().describe()

count    19015.000000
mean        38.828925
std         50.875735
min          1.000000
25%          8.000000
50%         23.000000
75%         49.000000
max        779.000000
dtype: float64

### 3.8.2. Исследование различий координат внутри почтового индекса


In [90]:
coordinates_stats = (geolocation.groupby("geolocation_zip_code_prefix", as_index=False)
       .agg(
           num_of_unique_lat=("geolocation_lat", "nunique"),
           num_of_unique_lng=("geolocation_lng", "nunique"),
       ))

coordinates_stats

,geolocation_zip_code_prefix,num_of_unique_lat,num_of_unique_lng
0,1001,10,10
1,1002,6,6
2,1003,10,10
3,1004,14,14
4,1005,11,11
...,...,...,...
19010,99960,5,5
19011,99965,6,6
19012,99970,16,16
19013,99980,21,21


In [91]:
unique_pairs = (
    geolocation[["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"]].drop_duplicates()
)
unique_pairs

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,1037,-23.545621,-46.639292
1,1046,-23.546081,-46.644820
2,1046,-23.546129,-46.642951
3,1041,-23.544392,-46.639499
4,1035,-23.541578,-46.641607
...,...,...,...
738327,99965,-28.180655,-52.034367
738328,99950,-28.072188,-52.011272
738329,99950,-28.068864,-52.012964
738330,99950,-28.068639,-52.010705


In [92]:
unique_pairs_count = unique_pairs.groupby("geolocation_zip_code_prefix").size().rename("num_of_unique_pairs").reset_index()
unique_pairs_count

,geolocation_zip_code_prefix,num_of_unique_pairs
0,1001,10
1,1002,6
2,1003,10
3,1004,14
4,1005,11
...,...,...
19010,99960,5
19011,99965,6
19012,99970,16
19013,99980,21


In [93]:
coordinates_stats = coordinates_stats.merge(
    unique_pairs_count,
    on="geolocation_zip_code_prefix",
    how="left",
    validate="1:1"
)
coordinates_stats

,geolocation_zip_code_prefix,num_of_unique_lat,num_of_unique_lng,num_of_unique_pairs
0,1001,10,10,10
1,1002,6,6,6
2,1003,10,10,10
3,1004,14,14,14
4,1005,11,11,11
...,...,...,...,...
19010,99960,5,5,5
19011,99965,6,6,6
19012,99970,16,16,16
19013,99980,21,21,21


In [94]:
#сколько индексов имеют ровно одну уникальную пару координат
coordinates_stats[coordinates_stats["num_of_unique_pairs"] == 1]

,geolocation_zip_code_prefix,num_of_unique_lat,num_of_unique_lng,num_of_unique_pairs
81,1142,1,1,1
90,1189,1,1,1
91,1200,1,1,1
151,1262,1,1,1
152,1290,1,1,1
...,...,...,...,...
18777,98320,1,1,1
18815,98620,1,1,1
18846,98825,1,1,1
18885,99043,1,1,1


In [95]:
#сколько индексов имеют несколько уникальных пар координат
coordinates_stats[coordinates_stats["num_of_unique_pairs"] > 1]

,geolocation_zip_code_prefix,num_of_unique_lat,num_of_unique_lng,num_of_unique_pairs
0,1001,10,10,10
1,1002,6,6,6
2,1003,10,10,10
3,1004,14,14,14
4,1005,11,11,11
...,...,...,...,...
19010,99960,5,5,5
19011,99965,6,6,6
19012,99970,16,16,16
19013,99980,21,21,21


### 3.8.3. Проверка городов и штатов внутри почтового индекса

In [96]:
city_and_states_num = geolocation.groupby("geolocation_zip_code_prefix", as_index=False).agg(
    num_of_unique_city=("geolocation_city", "nunique"),
    num_of_unique_state=("geolocation_state", "nunique"),
)
city_and_states_num

,geolocation_zip_code_prefix,num_of_unique_city,num_of_unique_state
0,1001,2,1
1,1002,2,1
2,1003,2,1
3,1004,2,1
4,1005,2,1
...,...,...,...
19010,99960,1,1
19011,99965,1,1
19012,99970,2,1
19013,99980,1,1


In [97]:
city_and_states_num[city_and_states_num["num_of_unique_state"] > 1]

,geolocation_zip_code_prefix,num_of_unique_city,num_of_unique_state
367,2116,2,2
1668,4011,2,2
6506,21550,1,2
6696,23056,1,2
14664,72915,3,2
15879,78557,2,2
16146,79750,1,2
16256,80630,2,2


В таблице `geolocation` для части почтовых индексов обнаружено несколько различных значений города. Это указывает на неоднозначность или вариативность текстового представления населённых пунктов в исходных данных. Кроме того, для небольшого числа почтовых индексов встречаются записи, относящиеся к двум различным штатам, что может свидетельствовать о неточностях или пограничных случаях в географических данных. Поэтому для дальнейшего анализа город и штат не будут определяться по таблице geolocation: эти признаки уже содержатся в таблицах `customers` и `sellers`. Таблица `geolocation` будет использоваться преимущественно для получения агрегированных координат по почтовому индексу.

### 3.8.4. Выбор способа агрегации координат

Будем использовать медиану широты и долготы для получение координат на один почтовый индекс. Внутри одного почтового индекса могут встречаться отдельные координаты, сильно отличающиеся от остальных. Медиана менее чувствительна к таким выбросам, чем среднее.

### 3.8.5. Предварительная агрегация до уровня почтового индекса


In [98]:
geolocation_agg = geolocation.groupby("geolocation_zip_code_prefix", as_index=False).agg(
    geolocation_lat=("geolocation_lat", "median"),
    geolocation_lng=("geolocation_lng", "median"),
    num_of_records_for_this_index=("geolocation_lng", "size")
)
geolocation_agg

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,num_of_records_for_this_index
0,1001,-23.549951,-46.634027,11
1,1002,-23.548228,-46.635247,6
2,1003,-23.548977,-46.635313,11
3,1004,-23.549550,-46.634771,14
4,1005,-23.549763,-46.636100,13
...,...,...,...,...
19010,99960,-27.953797,-52.029641,5
19011,99965,-28.179542,-52.035551,6
19012,99970,-28.343257,-51.875470,16
19013,99980,-28.388342,-51.846871,21


In [99]:
geolocation_agg = geolocation_agg.merge(
    unique_pairs_count,
    on="geolocation_zip_code_prefix",
    how="left",
    validate="1:1"
)
geolocation_agg

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,num_of_records_for_this_index,num_of_unique_pairs
0,1001,-23.549951,-46.634027,11,10
1,1002,-23.548228,-46.635247,6,6
2,1003,-23.548977,-46.635313,11,10
3,1004,-23.549550,-46.634771,14,14
4,1005,-23.549763,-46.636100,13,11
...,...,...,...,...,...
19010,99960,-27.953797,-52.029641,5,5
19011,99965,-28.179542,-52.035551,6,6
19012,99970,-28.343257,-51.875470,16,16
19013,99980,-28.388342,-51.846871,21,21


На первом этапе медианные координаты были рассчитаны по всем исходным географическим записям. Далее была выполнена проверка результата на географические аномалии.

### 3.8.6. Проверка предварительной агрегации

In [100]:
#уникален ли geolocation_zip_code_prefix?
geolocation_agg["geolocation_zip_code_prefix"].is_unique

True

In [101]:
#количество строк равно количеству уникальных почтовых индексов исходной geolocation?
assert len(geolocation_agg) == geolocation["geolocation_zip_code_prefix"].nunique()

In [102]:
#нет ли полных дубликатов?
geolocation_agg.duplicated().sum()

np.int64(0)

In [103]:
#появились ли пропуски в итоговой широте?
geolocation_agg["geolocation_lat"].isna().sum()

np.int64(0)

In [104]:
#появились ли пропуски в итоговой долготе?
geolocation_agg["geolocation_lng"].isna().sum()

np.int64(0)

In [105]:
#проверка: количество уникальных пар не может быть больше количества исходных строк
(
    geolocation_agg["num_of_unique_pairs"] <= geolocation_agg["num_of_records_for_this_index"]
).all()

np.True_

In [106]:
#для широты значения должны быть физически допустимыми: от -90 до 90
assert geolocation_agg["geolocation_lat"].between(-90, 90).all()

In [107]:
#для долготы значения должны быть физически допустимыми: от -180 до 180
assert geolocation_agg["geolocation_lng"].between(-180, 180).all()

### 3.8.7. Проверка координат на аномалии

In [108]:
geolocation_agg["geolocation_lat"].describe()

count    19015.000000
mean       -19.061457
std          7.293263
min        -33.690729
25%        -23.564480
50%        -22.423666
75%        -15.613478
max         42.184003
Name: geolocation_lat, dtype: float64

In [109]:
geolocation_agg["geolocation_lng"].describe()

count    19015.000000
mean       -46.059759
std          5.366919
min        -72.909444
25%        -49.007672
50%        -46.632679
75%        -43.252828
max        121.105394
Name: geolocation_lng, dtype: float64

In [110]:
outside_Brazil_geo = geolocation[~geolocation["geolocation_lat"].between(-34, 6) | ~geolocation["geolocation_lng"].between(-74, -28)]
outside_Brazil_geo

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
272046,18243,28.008978,-15.536867,bom retiro da esperanca,SP
355591,28165,41.614052,-8.411675,vila nova de campos,RJ
355603,28155,-34.586422,-58.732101,santa maria,RJ
355694,28155,42.439286,13.820214,santa maria,RJ
356233,28333,38.381672,-6.328200,raposo,RJ
358103,28595,43.684961,-7.411080,portela,RJ
374435,29654,29.409252,-98.484121,santo antônio do canaã,ES
374467,29654,21.657547,-101.466766,santo antonio do canaa,ES
409379,35179,25.995203,-98.078544,santana do paraíso,MG
409397,35179,25.995245,-98.078533,santana do paraiso,MG


In [111]:
outside_Brazil_geo["geolocation_zip_code_prefix"].nunique()

20

27 строк исходной таблицы `geolocation` выходят за диапазон, приближённо ограничивающий Бразилию. Они затрагивают 20 различных почтовых индексов.

In [112]:
anomalies_geo = geolocation_agg[~geolocation_agg["geolocation_lat"].between(-34, 6) | ~geolocation_agg["geolocation_lng"].between(-74, -28)]
anomalies_geo


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,num_of_records_for_this_index,num_of_unique_pairs
6077,18243,28.008978,-15.536867,1,1
10691,46560,13.001420,-23.583939,2,2
11738,57319,10.903855,-18.982951,2,2
13734,68275,19.836719,-30.411009,6,6
13748,68379,9.049063,-25.170903,2,2
15781,78131,38.816816,-9.394625,1,1
16419,83252,42.184003,-8.723762,1,1
18309,95130,14.585073,121.105394,1,1


Проверка предварительной `geolocation_agg` показала 8 почтовых индексов с аномальной итоговой координатой.

In [113]:
anomalous_zip_records = geolocation[geolocation["geolocation_zip_code_prefix"].isin(anomalies_geo["geolocation_zip_code_prefix"])]
anomalous_zip_records

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
272046,18243,28.008978,-15.536867,bom retiro da esperanca,SP
500380,46560,-12.989122,-42.220055,ibiajara,BA
500424,46560,38.991963,-4.947823,ibiajara,BA
524738,57319,-23.258224,-47.307430,vila sao francisco,AL
524889,57319,45.065933,9.341528,pau d'arco,AL
558526,68275,41.146203,-8.577855,porto trombetas,PA
558542,68275,42.166805,-6.898531,porto trombetas,PA
558546,68275,-1.743457,-52.244269,porto trombetas,PA
558558,68275,-1.743515,-52.244163,porto trombetas,PA
558581,68275,42.167251,-6.898559,porto trombetas,PA


In [114]:
anomalous_zip_records[anomalous_zip_records["geolocation_lat"].between(-34, 6) & anomalous_zip_records["geolocation_lng"].between(-74, -28) & (anomalous_zip_records["geolocation_zip_code_prefix"] == 68275)]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
558546,68275,-1.743457,-52.244269,porto trombetas,PA
558558,68275,-1.743515,-52.244163,porto trombetas,PA
558642,68275,-1.472765,-56.378018,oriximina,PA


In [115]:
# определяем, является ли каждая координата допустимой
anomalous_zip_records = anomalous_zip_records.copy()

anomalous_zip_records["is_valid_coordinate"] = (
    anomalous_zip_records["geolocation_lat"].between(-34, 6)
    & anomalous_zip_records["geolocation_lng"].between(-74, -28)
)

# создаём диагностическую сводку по каждому проблемному ZIP
anomalous_zip_summary = (
    anomalous_zip_records
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        total_records=("geolocation_zip_code_prefix", "size"),
        valid_records=("is_valid_coordinate", "sum")
    )
)

# считаем количество аномальных координат
anomalous_zip_summary["anomalous_records"] = (
    anomalous_zip_summary["total_records"]
    - anomalous_zip_summary["valid_records"]
)

# определяем, можно ли восстановить координаты ZIP
anomalous_zip_summary["result"] = np.where(
    anomalous_zip_summary["valid_records"] > 0,
    "можно восстановить",
    "восстановить нельзя"
)

anomalous_zip_summary

,geolocation_zip_code_prefix,total_records,valid_records,anomalous_records,result
0,18243,1,0,1,восстановить нельзя
1,46560,2,1,1,можно восстановить
2,57319,2,1,1,можно восстановить
3,68275,6,3,3,можно восстановить
4,68379,2,1,1,можно восстановить
5,78131,1,0,1,восстановить нельзя
6,83252,1,0,1,восстановить нельзя
7,95130,1,0,1,восстановить нельзя


Для 8 почтовых индексов с аномальными агрегированными координатами были исследованы исходные записи. Для индексов 46560, 57319, 68275 и 68379 обнаружены как корректные, так и аномальные координаты, поэтому их географическое положение можно восстановить после исключения ошибочных записей. Для индексов 18243, 78131, 83252 и 95130 ни одной допустимой координаты не найдено, поэтому восстановить их координаты по имеющимся данным невозможно. Для таких индексов итоговые координаты будут сохранены как пропуски.

In [116]:
geolocation_processed = geolocation.copy()

rows_before_filtration = len(geolocation_processed)
unique_zip_count_before = geolocation_processed["geolocation_zip_code_prefix"].nunique()

geolocation_processed["is_valid_coordinate"] = (
    geolocation_processed["geolocation_lat"].between(-34, 6)
    & geolocation_processed["geolocation_lng"].between(-74, -28)
)

#invalid_zip_count = geolocation_processed[geolocation_processed["is_valid_coordinate"] == False]["geolocation_zip_code_prefix"].nunique()
#print(invalid_zip_count)

geolocation_processed = geolocation_processed[geolocation_processed["is_valid_coordinate"] == True]

rows_after_filtration = len(geolocation_processed)
unique_zip_count_after = geolocation_processed["geolocation_zip_code_prefix"].nunique()

zip_before = set(geolocation["geolocation_zip_code_prefix"])
zip_after = set(geolocation_processed["geolocation_zip_code_prefix"])

zip_without_valid_coordinates = zip_before - zip_after

print("Строк до фильтрации: ", rows_before_filtration)
print("Строк после фильтрации: ", rows_after_filtration)
print("Аномальных строк исключено: ", rows_before_filtration - rows_after_filtration)
print("Уникальных zip до фильтрации: ", unique_zip_count_before)
print("Уникальных zip после фильтрации: ", unique_zip_count_after)

print("Индексы, которые были в исходной таблце, но отсутствуют в очищенной: ", zip_without_valid_coordinates)


Строк до фильтрации:  738332
Строк после фильтрации:  738305
Аномальных строк исключено:  27
Уникальных zip до фильтрации:  19015
Уникальных zip после фильтрации:  19011
Индексы, которые были в исходной таблце, но отсутствуют в очищенной:  {18243, 78131, 83252, 95130}


### 3.8.8. Финальная агрегация координат

In [117]:
#агрегация очищенных данных
geolocation_agg_clean = (
    geolocation_processed
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat=("geolocation_lat", "median"),
        geolocation_lng=("geolocation_lng", "median"),
        num_of_valid_records=("geolocation_zip_code_prefix", "size")
    )
)

#количество уникальных пар для каждого zip
valid_unique_pairs = (
    geolocation_processed[
        [
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng"
        ]
    ]
    .drop_duplicates()
)

valid_unique_pairs_count = (
    valid_unique_pairs
    .groupby("geolocation_zip_code_prefix")
    .size()
    .rename("num_of_unique_valid_pairs")
    .reset_index()
)

#присоединение valid_unique_pairs_count
geolocation_agg_clean = geolocation_agg_clean.merge(
    valid_unique_pairs_count,
    on="geolocation_zip_code_prefix",
    how="left",
    validate="1:1"
)
len(geolocation_agg_clean)

19011

In [118]:
#отдельная таблица ВСЕХ исходных ZIP
all_zip_codes = (
    geolocation[
        ["geolocation_zip_code_prefix"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

len(all_zip_codes)

19015

In [119]:
#финальная geolocation_agg
geolocation_agg = all_zip_codes.merge(
    geolocation_agg_clean,
    on="geolocation_zip_code_prefix",
    how="left",
    validate="1:1"
)
geolocation_agg

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,num_of_valid_records,num_of_unique_valid_pairs
0,1037,-23.545621,-46.639292,19.0,15.0
1,1046,-23.545742,-46.643199,37.0,29.0
2,1041,-23.544135,-46.639727,14.0,11.0
3,1035,-23.541578,-46.641607,15.0,12.0
4,1012,-23.547742,-46.634829,13.0,13.0
...,...,...,...,...,...
19010,99955,-28.107588,-52.145536,10.0,9.0
19011,99970,-28.343257,-51.875470,16.0,16.0
19012,99910,-27.860290,-52.084582,4.0,4.0
19013,99920,-27.854983,-52.300271,8.0,8.0


In [120]:
#заполнение счётчиков для invalid zip нулями
geolocation_agg[
    ["num_of_valid_records", "num_of_unique_valid_pairs"]
] = (
    geolocation_agg[
        ["num_of_valid_records", "num_of_unique_valid_pairs"]
    ]
    .fillna(0)
    .astype(int)
)

In [121]:
geolocation_agg[
    geolocation_agg["geolocation_zip_code_prefix"].isin(
        zip_without_valid_coordinates
    )
]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,num_of_valid_records,num_of_unique_valid_pairs
6095,18243,NaN,NaN,0,0
15802,78131,NaN,NaN,0,0
16421,83252,NaN,NaN,0,0
18328,95130,NaN,NaN,0,0


### 3.8.9. Проверка финальной географической таблицы


In [122]:
geolocation_agg.shape

(19015, 5)

In [123]:
assert len(geolocation_agg) == (
    geolocation["geolocation_zip_code_prefix"].nunique()
)

assert geolocation_agg[
    "geolocation_zip_code_prefix"
].is_unique

In [124]:
geolocation_agg[
    ["geolocation_lat", "geolocation_lng"]
].isna().sum()

geolocation_lat    4
geolocation_lng    4
dtype: int64

### 3.8.10. Добавление координат покупателям


In [125]:
customers_count_before = len(customers)

customers_geo = customers.merge(
    geolocation_agg,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="m:1"
)

customers_geo = customers_geo.rename(
    columns={
        "geolocation_lat": "customer_lat",
        "geolocation_lng": "customer_lng"
    }
)

In [126]:
#количество строк не изменилось
assert len(customers_geo) == customers_count_before

#customer_id остался уникальным
assert customers_geo["customer_id"].is_unique

#широта и долгота пропущены одновременно
assert (
    customers_geo["customer_lat"].isna()
    == customers_geo["customer_lng"].isna()
).all()

#сколько покупателей получили координаты
customers_geo["customer_lat"].notna().sum()

np.int64(99162)

In [127]:
#сколько покупателей осталось без координат
customers_geo["customer_lat"].isna().sum()

np.int64(279)

In [128]:
customers_geo = customers_geo.drop(
    columns=["geolocation_zip_code_prefix", "num_of_valid_records", "num_of_unique_valid_pairs"]
)

In [129]:
customers_geo

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,-20.502307,-47.396740
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,-23.730435,-46.541474
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,-23.531294,-46.656980
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,-23.499025,-46.183436
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,-22.974331,-47.142173
...,...,...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP,-23.586845,-46.499651
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP,-23.617326,-46.766974
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE,-3.735988,-38.510484
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS,-29.949783,-51.169077


### 3.8.11. Анализ покупателей без координат


In [130]:
customers_geo[
    customers_geo["customer_lat"].isna()
].drop_duplicates()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng
354,ecb1725b26e8b8c458181455dfa434ea,b55a113bb84fc10eaf58c6d09ec69794,72300,brasilia,DF,NaN,NaN
382,bcf86029aeed4ed8bac0e16eb14c22f5,7cd7974c9f79f75b77f323878ef87f43,11547,cubatao,SP,NaN,NaN
877,f4302056f0c58570522590f8181de2c7,67b05b597a66b5c449025000b9430abb,64605,picos,PI,NaN,NaN
1218,03bbe0ce5c28e05f22917607db798818,8f3dca4306d5a89e4ae2c65c110603a2,72465,brasilia,DF,NaN,NaN
1272,ad4950aded55c2ea376be59506456d68,aa2b96dd03307ea6dc4b763c0b5f0b39,7729,caieiras,SP,NaN,NaN
...,...,...,...,...,...,...,...
97467,cf818420383856a129134f5f8343f7b8,795c495a65f983b242fb01bd507977c5,72338,brasilia,DF,NaN,NaN
97780,67f3e907dce402e696b15f9308ff22ed,6f232f2f5c7f33b7bd9d794d2afacadd,68629,paragominas,PA,NaN,NaN
98140,f792e419335df11d82c32efcfb09c51b,c04c085b8e7573ba87b9ae1968d0985e,28530,sao sebastiao do paraiba,RJ,NaN,NaN
98878,78a11bb1fa72f556996b9a5b9bcd0629,e7536f62a200b415edd9491ac12a17fa,55863,siriji,PE,NaN,NaN


In [131]:
customers_geo[
    customers_geo["customer_zip_code_prefix"].isna()
]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng


In [132]:
customers_without_coordinates = customers_geo.loc[
    customers_geo["customer_lat"].isna()
].copy()

customers_without_coordinates["zip_exists_in_geolocation"] = (
    customers_without_coordinates["customer_zip_code_prefix"]
    .isin(geolocation["geolocation_zip_code_prefix"])
)

customers_without_coordinates["zip_exists_in_geolocation"].value_counts()

zip_exists_in_geolocation
False    278
True       1
Name: count, dtype: int64

In [133]:
customers_without_coordinates[
    customers_without_coordinates["customer_zip_code_prefix"].isin(
        zip_without_valid_coordinates
    )
][
    ["customer_zip_code_prefix", "customer_city", "customer_state"]
].drop_duplicates()

,customer_zip_code_prefix,customer_city,customer_state
84222,83252,ilha dos valadares,PR


Среди 279 покупателей без координат у 278 почтовый индекс полностью отсутствует в исходной таблице `geolocation`, поэтому определить их координаты по имеющимся данным невозможно. Только у одного покупателя ZIP (83252) присутствовал в geolocation, однако все доступные для него координаты были признаны аномальными и исключены при очистке. Таким образом, отсутствие координат в основном связано не с фильтрацией аномалий, а с неполным покрытием почтовых индексов таблицей geolocation.

### 3.8.12. Вывод

В ходе подготовки географических данных была исследована таблица `geolocation`. Было установлено, что одному почтовому индексу могут соответствовать несколько записей и несколько различных пар координат, поэтому прямое объединение этой таблицы с данными покупателей могло привести к размножению строк. Кроме того, для части почтовых индексов встречались различные значения города и в редких случаях штата. Поэтому таблица `geolocation` использовалась в дальнейшем только как источник координат, а информацию о городе и штате было решено брать непосредственно из таблицы `customers`.

Для получения одной географической записи на почтовый индекс координаты были агрегированы с использованием медианы широты и долготы. Медиана была выбрана как более устойчивый к отдельным выбросам показатель. Предварительная агрегация сформировала 19 015 уникальных почтовых индексов, однако последующая проверка показала, что для 8 индексов даже медианные координаты находились за пределами принятого допустимого диапазона для Бразилии.

Исследование исходных данных показало, что проблема связана с отдельными явно некорректными координатами. Всего было обнаружено 27 таких записей, относящихся к 20 почтовым индексам. Для большинства этих индексов наряду с аномальными присутствовали корректные координаты, поэтому после исключения ошибочных точек индекс сохранялся и его координаты можно было пересчитать. При этом для 4 почтовых индексов — `18243`, `78131`, `83252` и `95130` — ни одной допустимой координаты не осталось.

После исключения аномальных точек количество почтовых индексов с доступными координатами уменьшилось с 19 015 до 19 011. Затем медианные координаты были рассчитаны повторно только по корректным записям. Чтобы не терять четыре индекса без достоверных координат, в итоговой таблице `geolocation_agg` были сохранены все 19 015 исходных ZIP, а для невосстанавливаемых индексов значения широты и долготы оставлены как `NaN`.

Полученная географическая таблица была присоединена к `customers` по почтовому индексу связью `m:1`. После объединения количество покупателей не изменилось и составило 99 441, а строки не размножились. Координаты удалось определить для подавляющего большинства покупателей. Без координат осталось 279 покупателей: для 278 из них почтовый индекс вообще отсутствовал в исходной таблице `geolocation`, а ещё у одного покупателя индекс `83252` присутствовал, но все соответствующие ему координаты оказались аномальными и были исключены.

Таким образом, была сформирована очищенная и приведённая к необходимому уровню детализации географическая информация: **одна запись соответствует одному почтовому индексу**, явно ошибочные координаты не участвуют в расчётах, невозможные для достоверного восстановления значения сохранены как пропуски, а покупателям добавлены координаты без потери или размножения исходных наблюдений. Полученные данные готовы к дальнейшему формированию аналитической таблицы и географическому анализу заказов.


## 3.9. Сборка единой аналитической таблицы уровня заказа

### 3.9.1. Подготовка и контрольные показатели

In [134]:
#количество строк
orders_rows = len(orders)
#количество уникальных order_id
num_of_unique_order_id = orders["order_id"].nunique()
#количество столбцов
num_of_cols = orders.shape[1]

In [135]:
orders["order_id"].is_unique

True

| Таблица | Уровень детализации |
|---|---|
| `orders` | один заказ |
| `customers_geo` | один `customer_id` |
| `order_items_agg` | один заказ |
| `order_payments_agg` | один заказ с платежами |
| `order_reviews_final` | один заказ с отзывом |

### 3.9.2. Добавление данных покупателей

In [136]:
orders = orders.merge(
    customers_geo,
    on="customer_id",
    how="left",
    validate="m:1"
)

In [137]:
#проверка: количество строк не изменилось
assert orders_rows == len(orders)

#проверка: order_id всё ещё уникален
assert orders["order_id"].is_unique

#проверка: не появились ли дубли заказов
assert not orders.duplicated().any()

In [138]:
#проверка: сколько заказов не нашли соответствующего покупателя?
orders[orders["customer_unique_id"].isna()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng


Все заказы нашли соответствующего покупателя.

In [139]:
#проверка: появились ли новые пропуски в полях покупателя?
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
customer_lat                      279
customer_lng                      279
dtype: int64

Данные покупателей успешно присоединены к таблице заказов по `customer_id`. Количество строк не изменилось, `order_id` сохранил уникальность, размножения и дублирования заказов не произошло. Для всех заказов была найдена соответствующая запись покупателя. В основных полях покупателей (`customer_unique_id`, почтовый индекс, город и штат) пропуски отсутствуют. Пропуски в координатах покупателей являются ранее выявленными при подготовке географических данных и не возникли в результате объединения.

### 3.9.3. Добавление характеристик товаров

In [140]:
orders = orders.merge(
    order_items_agg,
    on="order_id",
    how="left",
    validate="1:1"
)

In [141]:
#проверка: количество строк не изменилось
assert orders_rows == len(orders)

#проверка: order_id всё ещё уникален
assert orders["order_id"].is_unique

In [142]:
#сколько заказов не нашли строки в order_items_agg?
orders[orders["items_count"].isna()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,max_item_price,total_freight,mean_freight,total_weight_g,total_volume_cm3,has_unknown_category,has_multiple_sellers,has_multiple_products,order_total,freight_ratio
266,8e24261a7e58791d10cb1bf9da94df5c,64a254d30eed42cd0e6c36dddb88adf0,unavailable,2017-11-16 15:09:28,2017-11-16 15:26:57,NaT,NaT,2017-12-05,41fc647b8c6bd979b1b6364b60471b50,89288,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
586,c272bcd21c287498b4883c7512019702,9582c5bbecc65eb568e2c1d839b5cba1,unavailable,2018-01-31 11:31:37,2018-01-31 14:23:50,NaT,NaT,2018-02-16,0e634b16e4c585acbd7b2e8276ce6677,11701,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
687,37553832a3a89c9b2db59701c357ca67,7607cd563696c27ede287e515812d528,unavailable,2017-08-14 17:38:02,2017-08-17 00:15:18,NaT,NaT,2017-09-05,596ed6d7a35890b3fbac54ec01f69685,2318,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
737,d57e15fb07fd180f06ab3926b39edcd2,470b93b3f1cde85550fc74cd3a476c78,unavailable,2018-01-08 19:39:03,2018-01-09 07:26:08,NaT,NaT,2018-02-06,2349bbb558908e0955e98d47dacb7adb,48607,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1130,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaT,NaT,NaT,2018-09-12,4fa4365000c7090fcb8cad5713c6d3db,1151,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99252,aaab15da689073f8f9aa978a390a69d1,df20748206e4b865b2f14a5eabbfcf34,unavailable,2018-01-16 14:27:59,2018-01-17 03:37:34,NaT,NaT,2018-02-06,a33e0969408919ba06779f497ead93ec,7025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99283,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,2018-08-31 16:13:44,NaT,NaT,NaT,2018-10-01,e90598185d2427a35e32ef241a5c04aa,11075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99347,a89abace0dcc01eeb267a9660b5ac126,2f0524a7b1b3845a1a57fcf3910c4333,canceled,2018-09-06 18:45:47,NaT,NaT,NaT,2018-09-27,d05c44a138277ad325d915c6b7ccbcdf,5344,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99348,a69ba794cc7deb415c3e15a0a3877e69,726f0894b5becdf952ea537d5266e543,unavailable,2017-08-23 16:28:04,2017-08-28 15:44:47,NaT,NaT,2017-09-15,e72a90a2b29fe1a8795b284aaaa3246f,22723,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [143]:
#какие товарные признаки получили NaN?
orders[["items_count", 
        "products_count", 
        "sellers_count", 
        "categories_count", 
        "unknown_category_items", 
        "total_price", 
        "mean_item_price", 
        "max_item_price",
        "total_freight", 
        "mean_freight", 
        "total_weight_g",
        "total_volume_cm3",
        "has_unknown_category",
        "has_multiple_sellers",
        "has_multiple_products",
        "order_total",
        "freight_ratio"]].isna().sum()

items_count               775
products_count            775
sellers_count             775
categories_count          775
unknown_category_items    775
total_price               775
mean_item_price           775
max_item_price            775
total_freight             775
mean_freight              775
total_weight_g            791
total_volume_cm3          791
has_unknown_category      775
has_multiple_sellers      775
has_multiple_products     775
order_total               775
freight_ratio             775
dtype: int64

In [144]:
orders[orders["items_count"].isna()]["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

Агрегированные характеристики товарных позиций были присоединены к таблице заказов по `order_id` со связью `1:1`. После объединения количество строк не изменилось, а `order_id` сохранил уникальность, следовательно, уровень детализации «одна строка - один заказ» не был нарушен.

Для `775` заказов соответствующие товарные позиции в `order_items_agg` отсутствуют. Большинство таких заказов имеют статусы `unavailable` (603) и `canceled` (164); также встречаются 5 заказов со статусом `created`, 2 - `invoiced` и 1 - `shipped`. Поэтому пропуски в основных агрегированных товарных признаках для этих 775 заказов являются следствием отсутствия записей о товарных позициях, а не ошибки объединения.

Для признаков общего веса и объёма обнаружено `791` пропущенное значение, то есть дополнительно у 16 заказов товарные позиции присутствуют, но необходимые физические характеристики товаров отсутствуют. Эти пропуски сохранены как неизвестные значения и не заменялись нулями.

### 3.9.4. Добавление платежных характеристик

In [145]:
orders = orders.merge(
    order_payments_agg,
    on="order_id",
    how="left",
    validate="1:1"
)

In [146]:
#проверка: количество строк не изменилось
assert orders_rows == len(orders)

#проверка: order_id всё ещё уникален
assert orders["order_id"].is_unique

In [147]:
#сколько заказов не нашли платеж?
orders[orders["payment_total"].isna()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,has_multiple_products,order_total,freight_ratio,payment_total,payments_count,payment_types_count,max_installments,main_payment_type,has_multiple_payment_types,used_installments
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,830d5b7aaa3b6f1e9ad63703bec97d23,14600,...,0.0,143.46,0.05918,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [148]:
orders[["payment_total", "payments_count", "payment_types_count", "max_installments", "main_payment_type", "has_multiple_payment_types", "used_installments"]].isna().sum()

payment_total                 1
payments_count                1
payment_types_count           1
max_installments              1
main_payment_type             1
has_multiple_payment_types    1
used_installments             1
dtype: int64

Агрегированные платежные характеристики были присоединены к таблице заказов по `order_id` со связью `1:1`. После объединения количество строк не изменилось, а `order_id` сохранил уникальность, поэтому уровень детализации «одна строка - один заказ» не был нарушен.

Для одного заказа соответствующая запись в таблице платежей отсутствует. Для него все агрегированные платежные признаки (payment_total, количество платежей, количество способов оплаты, максимальное число рассрочек, основной способ оплаты и дополнительные индикаторы) имеют пропущенные значения. При этом заказ имеет статус `delivered`, то есть отсутствие платежной информации является особенностью исходных данных, а не следствием статуса заказа или ошибки объединения.

Пропуски для этого заказа сохранены как NaN, поскольку отсутствие информации о платеже нельзя интерпретировать как нулевую сумму платежа.

### 3.9.5. Добавление информации об отзывах

In [149]:
orders = orders.merge(
    order_reviews_final,
    on="order_id",
    how="left",
    validate="1:1"
)

In [150]:
#проверка: количество строк не изменилось
assert orders_rows == len(orders)

#проверка: order_id всё ещё уникален
assert orders["order_id"].is_unique

In [151]:
#сколько заказов осталось без review_score?
orders[orders["review_score"].isna()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,has_multiple_payment_types,used_installments,review_id,review_score,review_creation_date,review_answer_timestamp,has_review_comment,has_review_title,reviews_count,had_multiple_reviews
16,403b97836b0c04a622354cf531062e5f,738b086814c6fcc74b8cc583f8516ee3,delivered,2018-01-02 19:00:43,2018-01-02 19:09:04,2018-01-03 18:19:09,2018-01-20 01:38:59,2018-02-06,6e26bbeaa107ec34112c64e1ee31c0f5,21381,...,0.0,1.0,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN
154,6942b8da583c2f9957e990d028607019,52006a9383bf149a4fb24226b173106f,shipped,2018-01-10 11:33:07,2018-01-11 02:32:30,2018-01-11 19:39:23,NaT,2018-02-07,528b011eb7fab3d59c336cc7248eed3a,38600,...,0.0,0.0,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN
311,4906eeadde5f70b308c20c4a8f20be02,4e7656e34357b93f14b40c6400ca3f6e,delivered,2017-12-08 04:45:26,2017-12-12 03:50:30,2017-12-12 17:43:21,2018-01-09 18:04:58,2018-01-03,ea870f4fdfd85ac98ab775b76efe3143,23065,...,0.0,0.0,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN
382,b7a4a9ecb1cd3ef6a3e36a48e200e3be,c3d8fc500d86b1c961ee144395c13a57,delivered,2017-05-19 18:13:54,2017-05-20 11:35:41,2017-05-30 12:43:50,2017-06-08 07:53:42,2017-06-16,367f4686d7112d69feed92b02a1775ed,88501,...,0.0,1.0,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN
390,59b32faedc12322c672e95ec3716d614,5baa82a2c45fa3220cb57d9881db3211,delivered,2018-06-27 11:10:11,2018-06-28 02:15:51,2018-06-28 14:57:00,2018-07-06 16:37:36,2018-07-26,c56d066e503008b8d0bf4204857c588d,97110,...,0.0,0.0,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98697,0c384d67524b5b92aa2fa6c8baa9a983,53421895d40d7df28d68c22ffa043355,delivered,2017-06-05 19:20:11,2017-06-05 19:30:18,2017-06-07 11:23:06,2017-06-13 14:09:21,2017-06-27,2f3aabaf3f5d1e8a6c49ac2fdee3cf3e,7153,...,0.0,0.0,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN
98781,906a6b0a96d89ee226e4977e99b80b9e,274a720e69d300bc7696c8570f8978fe,delivered,2017-08-28 15:14:21,2017-08-28 15:25:29,2017-08-31 15:25:01,2017-09-05 19:47:44,2017-09-18,e2226f481eef03f2876d1a27f0f4b1b0,11030,...,0.0,0.0,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN
99007,5333db16fe357175d39c82840dd3269d,7e008e5ec21e044fe30c34ec4e9d0747,delivered,2018-03-10 18:18:20,2018-03-13 04:08:22,2018-03-13 17:58:52,2018-04-03 15:32:52,2018-03-29,d06f106a141c540ca3b3cc0a3a8bba39,8141,...,0.0,0.0,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN
99134,2f2df159f26ddb73d55ee72372200d3e,86a7245fffe6a418ca1658a13ecc4531,delivered,2017-07-17 01:19:50,2017-07-17 01:30:09,2017-07-17 22:14:50,2017-07-26 09:44:00,2017-08-09,1d532194f005426adcdf6d969640d56e,35240,...,0.0,1.0,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN


In [152]:
#reviews_count и had_multiple_reviews согласованы между собой?
orders[orders["had_multiple_reviews"] == True]["reviews_count"].value_counts()

reviews_count
2.0    543
3.0      4
Name: count, dtype: int64

In [153]:
#корректен ли диапазон review_score?
assert orders["review_score"].dropna().between(1, 5).all()

Итоговая информация об отзывах была присоединена к таблице заказов по `order_id` со связью `1:1`. После объединения количество строк не изменилось, а `order_id` сохранил уникальность, поэтому уровень детализации «одна строка - один заказ» не был нарушен.

Для 98 673 заказов имеется итоговая оценка, а `768` заказов не имеют соответствующего отзыва. Эти пропуски являются ожидаемыми и отражают отсутствие отзывов в исходных данных.

Признаки повторных отзывов согласованы: 547 заказов имели более одной записи об отзыве, из них 543 заказа - по два отзыва и 4 заказа - по три. Для таких заказов ранее выбран один итоговый отзыв согласно принятому правилу, при этом количество исходных записей сохранено в признаке `reviews_count`.

Все имеющиеся значения `review_score` находятся в допустимом диапазоне от 1 до 5. Таким образом, информация об отзывах успешно добавлена в аналитическую таблицу без потери или размножения заказов.

### 3.9.6. Сводка по отсутствующим связанным данным

In [154]:
orders["customer_lat"].isna().sum() #заказы без координат покупателя

np.int64(279)

|Тип отсутствующих данных | Количество заказов |
|---|---|
|Без товарных данных | 775 |
|Без платежных данных | 1|
|Без отзывов | 768 |
|Без координат | 279 |

### 3.9.7. Проверка целостности итоговой таблицы

In [155]:
#Число строк
assert orders_rows == len(orders)

#Уникальность order_id
assert orders["order_id"].is_unique

#Полные дубликаты
assert not orders.duplicated().any()

#Количество уникальных заказов
assert len(orders) == orders["order_id"].nunique()

#Согласованность логических признаков

#products_count <= items_count
check = orders[["products_count", "items_count"]].dropna()

assert (
    check["products_count"] <= check["items_count"]
).all()

#sellers_count <= items_count
check = orders[["sellers_count", "items_count"]].dropna()

assert (
    check["sellers_count"] <= check["items_count"]
).all()

#если has_multiple_products = 1, то products_count > 1
mask = orders["products_count"].notna()

assert (
    orders.loc[mask, "has_multiple_products"]
    == (orders.loc[mask, "products_count"] > 1).astype(int)
).all()

#если has_multiple_sellers = 1, то sellers_count > 1
mask = orders["sellers_count"].notna()

assert (
    orders.loc[mask, "has_multiple_sellers"]
    == (orders.loc[mask, "sellers_count"] > 1).astype(int)
).all()

#если had_multiple_reviews = 1, то reviews_count > 1
mask = orders["reviews_count"].notna()

assert (
    orders.loc[mask, "had_multiple_reviews"]
    == (orders.loc[mask, "reviews_count"] > 1).astype(int)
).all()

### 3.9.8. Проверка финансовых показателей

In [156]:
#сколько заказов участвует в финансовом сравнении?
orders["order_total"].notna().sum()

np.int64(98666)

In [157]:
orders["payment_total"].notna().sum()

np.int64(99440)

In [158]:
#количество заказов, участвующих в финансовом сравнении
mask = (
    orders["order_total"].notna()
    & orders["payment_total"].notna()
)
mask.sum()

np.int64(98665)

In [159]:
orders["payment_difference"] = (
    orders["payment_total"] - orders["order_total"]
)

In [160]:
orders["payment_difference"].describe()

count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: payment_difference, dtype: float64

In [161]:
orders["abs_payment_difference"] = orders["payment_difference"].abs()

In [162]:
orders["abs_payment_difference"].describe()

count    98665.000000
mean         0.033162
std          1.129109
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: abs_payment_difference, dtype: float64

In [163]:
payments_match = (
    orders.loc[mask, "abs_payment_difference"] <= 0.01
)

In [164]:
payments_match.value_counts()

abs_payment_difference
True     98284
False      381
Name: count, dtype: int64

In [165]:
#процент совпадающих заказов
payments_match.mean() * 100

np.float64(99.61384482845995)

In [166]:
orders.loc[
    mask,
    [
        "order_id",
        "order_status",
        "total_price",
        "total_freight",
        "order_total",
        "payment_total",
        "payment_difference",
        "payments_count",
        "payment_types_count",
        "main_payment_type",
        "max_installments"
    ]
].sort_values(
    "payment_difference",
    key=lambda x: x.abs(),
    ascending=False
).head(20)

,order_id,order_status,total_price,total_freight,order_total,payment_total,payment_difference,payments_count,payment_types_count,main_payment_type,max_installments
11791,ce6d150fb29ada17d2082f4847107665,delivered,1299.00,104.66,1403.66,1586.47,182.81,1.0,1.0,credit_card,10.0
48686,6e5fe7366a2e1bfbf3257dba0af1267f,delivered,179.19,108.72,287.91,406.92,119.01,1.0,1.0,credit_card,10.0
70865,70b742795bc441e94a44a084b6d9ce7a,delivered,269.99,196.94,466.93,578.82,111.89,1.0,1.0,credit_card,20.0
33150,996c7e73600ad3723e8627ab7bef81e4,delivered,559.90,28.00,587.90,664.43,76.53,1.0,1.0,credit_card,10.0
52985,70b7e94ea46d3e8b5bc12a50186edaf0,delivered,167.88,45.27,213.15,274.84,61.69,1.0,1.0,credit_card,24.0
85434,bc2c82b0ef78d2252b6176d1972db7c9,delivered,165.00,77.01,242.01,303.02,61.01,1.0,1.0,credit_card,21.0
94304,af9ffff2ce6b3defd34fd4c78857a379,delivered,395.65,17.52,413.17,466.97,53.80,1.0,1.0,credit_card,10.0
31661,262118ce178bb3e4590a3adcf6d62e6b,delivered,119.80,57.94,177.74,126.12,-51.62,1.0,1.0,credit_card,5.0
68811,bfdb5bbb06458d600a33d61f5f287472,delivered,297.00,51.93,348.93,394.36,45.43,1.0,1.0,credit_card,10.0
8548,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,delivered,209.80,44.65,254.45,293.89,39.44,1.0,1.0,credit_card,12.0


Для проверки согласованности финансовых показателей были отобраны `98 665` заказов, для которых одновременно известны рассчитанная стоимость заказа (`order_total`) и фактически уплаченная сумма (`payment_total`). Для большинства заказов показатели практически полностью совпадают: при допустимом расхождении не более `0,01` совпадение наблюдается для `98 284` заказов, то есть примерно для 99,61% рассматриваемых заказов. Расхождение выявлено только для `381` заказа.

Медиана абсолютного расхождения равна 0, как и значения первого и третьего квартилей, что подтверждает отсутствие систематических различий между рассчитанной стоимостью заказа и суммой платежей. Среднее абсолютное отклонение также невелико и составляет около 0,03. При этом присутствуют отдельные существенно отличающиеся случаи: максимальное абсолютное расхождение достигает 182,81, а диапазон исходной разницы payment_total - order_total составляет от −51,62 до 182,81.

Просмотр заказов с наибольшими расхождениями показал, что они не связаны исключительно с разделением оплаты между несколькими платежами или способами оплаты: среди наиболее крупных отклонений встречаются заказы с одним платежом, одним способом оплаты и оплатой банковской картой. При этом многие из них имеют большое число рассрочных платежей (max_installments), однако на основании текущей проверки нельзя однозначно утверждать, что именно рассрочка является причиной расхождений.

Таким образом, товарные и платежные показатели в итоговой таблице в целом хорошо согласованы, а обнаруженные финансовые расхождения относятся к небольшой группе заказов и могут быть дополнительно исследованы на этапе

### 3.9.9. Проверка структуры итоговой таблицы

In [167]:
#размер таблицы
orders.shape

(99441, 48)

In [168]:
#список столбцов
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'customer_lat', 'customer_lng', 'items_count',
       'products_count', 'sellers_count', 'categories_count',
       'unknown_category_items', 'total_price', 'mean_item_price',
       'max_item_price', 'total_freight', 'mean_freight', 'total_weight_g',
       'total_volume_cm3', 'has_unknown_category', 'has_multiple_sellers',
       'has_multiple_products', 'order_total', 'freight_ratio',
       'payment_total', 'payments_count', 'payment_types_count',
       'max_installments', 'main_payment_type', 'has_multiple_payment_types',
       'used_installments', 'review_id', 'review_score',
       'review_creation_date', 'review_answer_timestamp', 'has_review_comment',
       'has_review_

In [169]:
#типы данных
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 48 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
 8   customer_unique_id             99441 non-null  str           
 9   customer_zip_code_prefix       99441 non-null  int64         
 10  customer_city                  99441 non-null  str           
 11  customer_state            

In [170]:
#пропуски
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
customer_lat                      279
customer_lng                      279
items_count                       775
products_count                    775
sellers_count                     775
categories_count                  775
unknown_category_items            775
total_price                       775
mean_item_price                   775
max_item_price                    775
total_freight                     775
mean_freight                      775
total_weight_g                    791
total_volume_cm3                  791
has_unknown_

В результате последовательного объединения данных сформирована итоговая аналитическая таблица уровня заказа размером 99 441 строка и 48 столбцов. Каждая строка соответствует одному заказу и содержит исходные характеристики заказа и покупателя, агрегированные товарные и платёжные показатели, информацию об итоговом отзыве и географические признаки.

В ключевых идентификаторах и основных характеристиках заказа и покупателя пропуски отсутствуют. Обнаруженные пропуски в дополнительных признаках согласуются с результатами предыдущих этапов: 775 заказов не имеют связанных товарных данных, 768 — отзывов, 279 покупателей не получили географические координаты, а один заказ не имеет платёжной информации. Для некоторых заказов также отсутствуют отдельные даты жизненного цикла заказа и физические характеристики товаров.

Таким образом, структура итоговой таблицы соответствует выбранному уровню детализации «одна строка — один заказ», а имеющиеся пропуски имеют известное происхождение и не являются следствием некорректного объединения таблиц.

### 3.9.10. Сохранение итоговой аналитической таблицы

In [171]:
orders_analytics = orders.copy()

## 3.10. Создание временных признаков

In [172]:
orders_analytics_rows_before = len(orders_analytics)

### 3.10.1. Признаки момента покупки

In [173]:
orders_analytics["purchase_year"] = orders_analytics["order_purchase_timestamp"].dt.year
orders_analytics["purchase_month"] = orders_analytics["order_purchase_timestamp"].dt.month
orders_analytics["purchase_year_month"] = (
    orders_analytics["order_purchase_timestamp"]
    .dt.to_period("M")
)
orders_analytics["purchase_day_of_week"] = orders_analytics["order_purchase_timestamp"].dt.day_of_week
orders_analytics["purchase_hour"] = orders_analytics["order_purchase_timestamp"].dt.hour
orders_analytics["purchase_is_weekend"] = (orders_analytics["purchase_day_of_week"].isin([5, 6]))


In [174]:
orders_analytics[
    [
        "purchase_year",
        "purchase_month",
        "purchase_year_month",
        "purchase_day_of_week",
        "purchase_hour",
        "purchase_is_weekend"
    ]
].head()

,purchase_year,purchase_month,purchase_year_month,purchase_day_of_week,purchase_hour,purchase_is_weekend
0,2017,10,2017-10,0,10,False
1,2018,7,2018-07,1,20,False
2,2018,8,2018-08,2,8,False
3,2017,11,2017-11,5,19,True
4,2018,2,2018-02,1,21,False


### 3.10.2. Длительности этапов выполнения заказа

In [175]:
orders_analytics["approval_time_days"] = (orders_analytics["order_approved_at"] - orders_analytics["order_purchase_timestamp"]).dt.total_seconds() / 86400

#длительность между подтверждением и передачей в доставку
orders_analytics["carrier_handoff_time_days"] = (orders_analytics["order_delivered_carrier_date"] - orders_analytics["order_approved_at"]).dt.total_seconds() / 86400
#флаг некорректной последовательности
orders_analytics["invalid_approval_carrier_order"] = (
    orders_analytics["carrier_handoff_time_days"] < 0
)
#отрицательные длительности заменяем на nan
orders_analytics.loc[
    orders_analytics["invalid_approval_carrier_order"],
    "carrier_handoff_time_days"
] = np.nan

In [176]:
orders_analytics["invalid_approval_carrier_order"].sum()

np.int64(1359)

In [177]:
orders_analytics["delivery_time_days"] = (orders_analytics["order_delivered_customer_date"] - orders_analytics["order_purchase_timestamp"]).dt.total_seconds() / 86400
orders_analytics["estimated_delivery_time_days"] = (orders_analytics["order_estimated_delivery_date"] - orders_analytics["order_purchase_timestamp"]).dt.total_seconds() / 86400


In [178]:
(orders_analytics["carrier_handoff_time_days"] < 0).sum()

np.int64(0)

In [179]:
(orders_analytics["estimated_delivery_time_days"] < 0).sum()

np.int64(0)

In [180]:
(orders_analytics["approval_time_days"] < 0).sum()

np.int64(0)

In [181]:
(orders_analytics["delivery_time_days"] < 0).sum()

np.int64(0)

In [182]:
orders_analytics[
    [
        "approval_time_days",
        "carrier_handoff_time_days",
        "delivery_time_days",
        "estimated_delivery_time_days"
    ]
].describe()

,approval_time_days,carrier_handoff_time_days,delivery_time_days,estimated_delivery_time_days
count,99281.000000,96285.000000,96476.000000,99441.000000
mean,0.434129,2.859186,12.558702,23.767650
std,1.084917,3.498574,9.546530,8.832371
min,0.000000,0.000174,0.533414,1.648993
25%,0.008958,0.901539,6.766403,18.331690
50%,0.014306,1.851667,10.217755,23.240370
75%,0.607535,3.624051,15.720327,28.424861
max,187.882523,125.762569,209.628611,155.135463


Для заказов рассчитаны четыре временных признака, характеризующие основные этапы выполнения заказа. В среднем подтверждение заказа происходит через 0,43 дня, передача перевозчику — примерно через 2,86 дня после подтверждения, а фактическая доставка занимает около 12,56 дня с момента покупки. При этом планируемый срок доставки в среднем составляет 23,77 дня, то есть фактическая доставка обычно происходит заметно раньше установленного срока.

Для этапа между подтверждением заказа и передачей перевозчику ранее выявленные 1359 случаев некорректной последовательности дат были сохранены отдельным признаком, а соответствующие отрицательные длительности заменены на NaN. После обработки отрицательных значений в рассчитанных длительностях не осталось.

### 3.10.3. Признаки соблюдения срока доставки

In [183]:
#положительное значение - опоздание, отрицательное - доставка раньше срока
orders_analytics["delivery_delay_days"] = (orders_analytics["order_delivered_customer_date"] - orders_analytics["order_estimated_delivery_date"]).dt.total_seconds() / 86400

In [184]:
orders_analytics["delivery_delay_days"].describe()

count    96476.000000
mean       -11.179120
std         10.186113
min       -146.016123
25%        -16.244384
50%        -11.948941
75%         -6.390000
max        188.975081
Name: delivery_delay_days, dtype: float64

In [185]:
orders_analytics["is_late_delivery"] = (
    orders_analytics["delivery_delay_days"] > 0
).where(
    orders_analytics["delivery_delay_days"].notna()
)

In [186]:
#количество дней , на которое заказ опоздал
orders_analytics["delay_days"] = (
    orders_analytics["delivery_delay_days"].clip(lower=0)
)

#количество дней, на которое раньше доставили
orders_analytics["early_delivery_days"] = (
    (-orders_analytics["delivery_delay_days"]).clip(lower=0)
)

Созданы признаки соблюдения срока доставки. Рассчитано отклонение фактической даты доставки от обещанной (`delivery_delay_days`), а также производные признаки: факт опоздания (`is_late_delivery`), количество дней опоздания (`delay_days`) и количество дней досрочной доставки (`early_delivery_days`). Пропуски сохранены для заказов, по которым невозможно корректно определить срок доставки.

### 3.10.4. Проверка созданных временных признаков

In [187]:
#число строк не изменилось
assert len(orders_analytics) == orders_analytics_rows_before

In [188]:
#проверка: календарные признаки покупки заполнены у всех заказов
orders_analytics[
    [
        "purchase_year",
        "purchase_month",
        "purchase_year_month",
        "purchase_day_of_week",
        "purchase_hour",
        "purchase_is_weekend"
    ]
].isna().sum()

purchase_year           0
purchase_month          0
purchase_year_month     0
purchase_day_of_week    0
purchase_hour           0
purchase_is_weekend     0
dtype: int64

In [189]:
#проверка: NaN появляются именно из-за отсутствующих исходных дат
approval_source_missing = (
    orders_analytics[
        ["order_purchase_timestamp", "order_approved_at"]
    ]
    .isna()
    .any(axis=1)
)

assert (
    orders_analytics["approval_time_days"].isna()
    == approval_source_missing
).all()

In [190]:
delivery_source_missing = (
    orders_analytics[
        ["order_purchase_timestamp", "order_delivered_customer_date"]
    ]
    .isna()
    .any(axis=1)
)

assert (
    orders_analytics["delivery_time_days"].isna()
    == delivery_source_missing
).all()

In [191]:
estimated_source_missing = (
    orders_analytics[
        ["order_purchase_timestamp", "order_estimated_delivery_date"]
    ]
    .isna()
    .any(axis=1)
)

assert (
    orders_analytics["estimated_delivery_time_days"].isna()
    == estimated_source_missing
).all()

In [192]:
carrier_source_missing = (
    orders_analytics[
        ["order_approved_at", "order_delivered_carrier_date"]
    ]
    .isna()
    .any(axis=1)
)

expected_carrier_nan = (
    carrier_source_missing
    | orders_analytics["invalid_approval_carrier_order"]
)

assert (
    orders_analytics["carrier_handoff_time_days"].isna()
    == expected_carrier_nan
).all()

In [193]:
#если is_late_delivery = 1, разница должна быть положительной
mask = orders_analytics["delivery_delay_days"].notna()

assert (
    orders_analytics.loc[mask, "is_late_delivery"]
    ==
    (orders_analytics.loc[mask, "delivery_delay_days"] > 0)
).all()

assert (
    orders_analytics["is_late_delivery"].isna()
    ==
    orders_analytics["delivery_delay_days"].isna()
).all()

In [194]:
#проверка: delay_days и early_delivery_days не должны одновременно быть больше нуля.
assert not (
    (orders_analytics["delay_days"] > 0)
    & (orders_analytics["early_delivery_days"] > 0)
).any()

### 3.10.5. Итог по временным признакам


В рамках данного раздела были созданы временные признаки, характеризующие момент оформления заказа, длительность отдельных этапов его выполнения и соблюдение ожидаемого срока доставки.

| Признак | Смысл |
|---|---|
| `purchase_year` | Год оформления заказа |
| `purchase_month` | Месяц оформления заказа |
| `purchase_year_month` | Год и месяц оформления заказа для анализа динамики во времени |
| `purchase_day_of_week` | День недели оформления заказа: от 0 (понедельник) до 6 (воскресенье) |
| `purchase_hour` | Час оформления заказа |
| `purchase_is_weekend` | Признак оформления заказа в выходной день |
| `approval_time_days` | Время от оформления заказа до его подтверждения, в днях |
| `carrier_handoff_time_days` | Время от подтверждения заказа до передачи перевозчику, в днях |
| `invalid_approval_carrier_order` | Признак некорректной последовательности дат, когда передача перевозчику указана раньше подтверждения заказа |
| `delivery_time_days` | Фактическое время от оформления заказа до доставки покупателю, в днях |
| `estimated_delivery_time_days` | Планируемое время от оформления заказа до ожидаемой даты доставки, в днях |
| `delivery_delay_days` | Отклонение фактической даты доставки от ожидаемой: положительное значение означает опоздание, отрицательное — доставку раньше срока |
| `is_late_delivery` | Признак того, что заказ был доставлен позже ожидаемой даты |
| `delay_days` | Количество дней опоздания; для заказов без опоздания равно 0 |
| `early_delivery_days` | Количество дней, на которое заказ был доставлен раньше ожидаемого срока |

При расчёте длительностей пропуски сохранены в тех случаях, когда отсутствуют необходимые исходные даты. Для ранее выявленных некорректных последовательностей `approval → carrier` соответствующая длительность заменена на `NaN`, а информация об аномалии сохранена в отдельном признаке `invalid_approval_carrier_order`.

Таким образом, в аналитическую таблицу добавлен набор временных признаков, который можно использовать на следующих этапах EDA для анализа сезонности заказов, скорости прохождения этапов выполнения заказа и соблюдения сроков доставки.

## 3.11. Создание целевого признака

In [195]:
rows_before_bad_review = len(orders_analytics)

### 3.11.1. Определение целевого признака

Целевой признак `bad_review` характеризует наличие негативной оценки заказа:

bad_review = 1 — итоговая оценка заказа равна 1 или 2;

bad_review = 0 — итоговая оценка заказа равна 3, 4 или 5.

Для заказов без отзыва значение целевого признака не определяется.

### 3.11.2. Создание целевого признака

In [196]:
orders_analytics["bad_review"] = np.nan

orders_analytics.loc[
    orders_analytics["review_score"].between(1, 2),
    "bad_review"
] = 1

orders_analytics.loc[
    orders_analytics["review_score"].between(3, 5),
    "bad_review"
] = 0

### 3.11.3. Проверка корректности созданного таргета

In [197]:
#проверка: Для review_score 1–2 bad_review всегда равен 1.
assert (orders_analytics["review_score"].isin([1, 2]) == (orders_analytics["bad_review"] == 1)).all()

#проверка: Для review_score 3–5 bad_review всегда равен 0
assert (orders_analytics["review_score"].isin([3, 4, 5]) == (orders_analytics["bad_review"] == 0)).all()

#проверка: Если review_score отсутствует, bad_review тоже отсутствует.
assert (orders_analytics["review_score"].isna() == orders_analytics["bad_review"].isna()).all()

#проверка: Количество строк таблицы после создания признака не изменилось.
assert len(orders_analytics) == rows_before_bad_review


### 3.11.4. Распределение целевого признака

In [198]:
orders_analytics[
    ["review_score", "bad_review"]
].value_counts(dropna=False)

review_score  bad_review
5.0           0.0           57008
4.0           0.0           19038
1.0           1.0           11363
3.0           0.0            8133
2.0           1.0            3131
NaN           NaN             768
Name: count, dtype: int64

In [199]:
orders_analytics["bad_review"].value_counts(normalize=True) * 100

bad_review
0.0    85.311078
1.0    14.688922
Name: proportion, dtype: float64

Среди заказов с известной оценкой около `14,7%` относятся к классу плохих отзывов, а около `85,3%` — к классу неплохих отзывов. Таким образом, целевой признак заметно несбалансирован, что необходимо учитывать при дальнейшем выборе метрик и построении модели.

### 3.11.5. Сохранение обновлённой orders_analytics

In [200]:
orders_analytics.to_csv(
    "../data/processed/orders_analytics.csv",
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

### 3.11.6. Примечание о целевой переменной

Целевой признак `bad_review` непосредственно формируется на основе `review_score`. Поэтому `review_score` не должен использоваться в дальнейшем как входной признак модели.

Также признаки, непосредственно связанные с отзывом (`review_id`, `review_creation_date`, `review_answer_timestamp`, `has_review_comment`, `has_review_title`, `reviews_count`, `had_multiple_reviews`), необходимо отдельно проверить на возможность использования при моделировании, поскольку они могут содержать информацию, недоступную на момент прогнозирования, и приводить к утечке целевой переменной.